# Iceberg Bucket Partitioning

This notebook demonstrates **bucket partitioning** in Apache Iceberg using a TPC-H inspired `orders` dataset.

Here we partition `orders` by `bucket(8, customer_id)`.

It is used to test the new Iceberg connector of `df-executor`

### 1. Start Spark

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Jupyter").getOrCreate()
spark

26/04/15 12:14:16 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### 2. Create namespace

In [2]:
%%sql

CREATE NAMESPACE IF NOT EXISTS demo.tpch;

++
||
++
++

### 3. Create table with bucket partitioning

The `PARTITIONED BY (bucket(8, customer_id))` clause tells Iceberg to hash `customer_id` into 8 buckets.  
The partition transform is **invisible at query time**, so no need to filter on a partition column explicitly.

In [3]:
%%sql

DROP TABLE IF EXISTS demo.tpch.orders;

++
||
++
++

In [4]:
%%sql

CREATE TABLE demo.tpch.orders (
    order_id     BIGINT,
    customer_id  BIGINT,
    status       STRING,
    total_price  DOUBLE,
    order_date   DATE,
    priority     STRING
) USING iceberg
PARTITIONED BY (bucket(8, customer_id));

++
||
++
++

### 4. Populate with TPC-H inspired data

In [5]:
%%sql

INSERT INTO demo.tpch.orders VALUES
    (1,  370, 'O', 172799.49, DATE '1996-01-02', '5-LOW'),
    (2,  781, 'O', 38426.09,  DATE '1996-12-01', '1-URGENT'),
    (3,  1234,'F', 205654.30, DATE '1993-10-14', '5-LOW'),
    (4,  1369,'O', 56000.91,  DATE '1995-10-11', '5-LOW'),
    (5,  445, 'F', 105367.67, DATE '1994-07-30', '5-LOW'),
    (6,  55,  'F', 25022.40,  DATE '1992-02-21', '4-NOT SPECIFIED'),
    (7,  39,  'O', 231040.44, DATE '1996-01-10', '2-HIGH'),
    (8,  370, 'O', 131462.81, DATE '1995-10-04', '1-URGENT'),
    (9,  781, 'F', 84263.94,  DATE '1993-01-13', '3-MEDIUM'),
    (10, 1234,'P', 65522.05,  DATE '1998-07-21', '2-HIGH'),
    (11, 445, 'O', 18956.30,  DATE '1997-03-08', '1-URGENT'),
    (12, 55,  'F', 297575.57, DATE '1993-11-20', '4-NOT SPECIFIED'),
    (13, 39,  'F', 152245.70, DATE '1993-11-20', '3-MEDIUM'),
    (14, 1369,'O', 47555.20,  DATE '1998-01-15', '5-LOW'),
    (15, 370, 'F', 72800.99,  DATE '1994-04-05', '2-HIGH');

++
||
++
++

### 5. Inspect the partition layout

Iceberg tracks partition metadata in its own metadata tables.  
We can see how rows were distributed across the 8 buckets.

In [6]:
%%sql

SELECT
    partition,
    record_count,
    file_count
FROM demo.tpch.orders.partitions
ORDER BY partition;

partition,record_count,file_count
Row(customer_id_bucket=1),4,1
Row(customer_id_bucket=2),3,1
Row(customer_id_bucket=3),4,1
Row(customer_id_bucket=5),4,1


In [17]:
%%sql

-- select count(*) from demo.tpch.orders.files;
select * from demo.tpch.orders.snapshots;

committed_at,snapshot_id,parent_id,operation,manifest_list,summary
2026-04-10 08:36:24.275000,2510204552283908191,None,append,s3://warehouse/tpch/orders/metadata/snap-2510204552283908191-1-083e8525-33db-42d4-85af-fa72e7ee50db.avro,"{'engine-version': '3.5.5', 'added-data-files': '4', 'total-equality-deletes': '0', 'app-id': 'local-1775810181348', 'added-records': '15', 'total-records': '15', 'spark.app.id': 'local-1775810181348', 'changed-partition-count': '4', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '7917', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '7917', 'total-data-files': '4'}"


### 6. Query : Iceberg prunes buckets automatically

Filtering on `customer_id` lets Iceberg compute the target bucket hash and skip all other buckets.

In [3]:
%%sql

SELECT
    order_id,
    customer_id,
    status,
    total_price,
    order_date,
    priority
FROM demo.tpch.orders
WHERE customer_id = 370
ORDER BY order_date;

order_id,customer_id,status,total_price,order_date,priority
15,370,F,72800.99,1994-04-05,2-HIGH
8,370,O,131462.81,1995-10-04,1-URGENT
1,370,O,172799.49,1996-01-02,5-LOW


### 7. Aggregation per customer

In [4]:
%%sql

SELECT
    customer_id,
    COUNT(*)          AS order_count,
    SUM(total_price)  AS total_spent,
    MIN(order_date)   AS first_order,
    MAX(order_date)   AS last_order
FROM demo.tpch.orders
GROUP BY customer_id
ORDER BY total_spent DESC;

customer_id,order_count,total_spent,first_order,last_order
39,2,383286.14,1993-11-20,1996-01-10
370,3,377063.29,1994-04-05,1996-01-02
55,2,322597.97000000003,1992-02-21,1993-11-20
1234,2,271176.35,1993-10-14,1998-07-21
445,2,124323.97,1994-07-30,1997-03-08
781,2,122690.03,1993-01-13,1996-12-01
1369,2,103556.11,1995-10-11,1998-01-15


In [5]:
%%sql

SELECT
    customer_id,
    COUNT(*),
    SUM(total_price)
FROM demo.tpch.orders
GROUP BY customer_id

customer_id,count(1),sum(total_price)
39,2,383286.14
370,3,377063.29
55,2,322597.97000000003
1234,2,271176.35
1369,2,103556.11
445,2,124323.97
781,2,122690.03
